# 04 \u00b7 Live demo

Interactive debate demo for the presentation. It builds **one** `ArguBotPipeline` and keeps **one**
`ClaimTracker` alive across turns, so the bot argues consistently with what it said earlier \u2014 and
beside every reply it prints the **claim log** (what the bot has committed to) and the **retrieved
evidence**, so the audience can see the mechanism, not just the output.

Defaults to **System D** (retrieval + claim tracking + stance filter + 5-candidate NLI reranker).
Run it in `mynotebook` on the cluster with the trained adapter and passages present; set `DUMMY = True`
in the build cell for a quick dry run with no models or data.

## Setup

Finds the repo root, puts `src/` on the path, and installs missing deps (incl. flash-linear-attention).

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

# Colab check that does NOT raise when the `google` package is absent (it does on the
# SCIS cluster / a local venv). find_spec on a dotted name raises if the parent is missing.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# On Colab, clone the repo once. Elsewhere (cluster / local) it is already present.
if IN_COLAB:
    REPO_URL = "https://github.com/diyadj/argubot.git"
    if not os.path.isdir("argubot"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("argubot")

# Work from the repo root so configs/config.yaml and data/... resolve. Walk up from the launch
# dir (notebooks/ on the cluster), and fall back to ~/argubot (execnotebook runs from your home).
root = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "pyproject.toml").exists()), None)
if root is None and (Path.home() / "argubot" / "pyproject.toml").exists():
    root = Path.home() / "argubot"
if root is not None:
    os.chdir(root)

src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)

pip_name = {"torch": "torch", "transformers": "transformers", "peft": "peft",
            "accelerate": "accelerate", "numpy": "numpy", "yaml": "pyyaml"}
need = [pip_name[m] for m in pip_name if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
if importlib.util.find_spec("fla") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-linear-attention"], check=False)

for _m in [k for k in list(sys.modules) if k == "argubot" or k.startswith("argubot.")]:
    del sys.modules[_m]

import argubot  # noqa: F401
print("cwd:", os.getcwd(), "| IN_COLAB:", IN_COLAB, "| argubot:", argubot.__file__)

## Build the pipeline

Loads the components once and creates the single pipeline + tracker the whole demo reuses.

In [ ]:
from argubot.config import load_config
from argubot.systems import ArguBotPipeline, build_systems
from argubot.systems.components import load_components
from argubot.claims.tracker import ClaimTracker
from argubot.schemas import Turn

cfg = load_config("configs/config.yaml")

# --- demo settings ---
SYSTEM = "D"    # full system: retrieval + claim log + stance filter + 5-candidate NLI reranker
DUMMY = False   # True = fake generator/retriever/NLI (no models or data) for a quick dry run

# Real components load the trained adapter (outputs/adapter), the retriever (passages) and the NLI
# model; DUMMY swaps in canned stand-ins so the loop can be tested without any of that.
generator, retriever, nli = load_components(cfg, dummy=DUMMY)
systems = build_systems(cfg["decoding"]["n_candidates_rerank"])
pipeline = ArguBotPipeline(systems[SYSTEM], generator, cfg, retriever=retriever, nli=nli)

# ONE tracker and ONE history, kept alive across turns -- that persistence is the whole point:
# the bot sees what it argued earlier, so the audience can watch the claim log grow and the
# replies stay consistent with it.
tracker = ClaimTracker(max_claims=cfg["claims"]["max_claims"], dedup_threshold=cfg["claims"]["dedup_threshold"])
history, turn_index = [], 0
print(f"Ready. System {SYSTEM}" + (" (DUMMY components)" if DUMMY else "") + f" | claim log keeps {cfg['claims']['max_claims']} claims")

## The turn function

`respond(text)` runs one user turn through the shared pipeline + tracker and prints the reply, the
evidence used, and the running claim log.

In [ ]:
import textwrap

def _show(label, items):
    print(f"  {label}:")
    for it in (items or []):
        print("    - " + textwrap.fill(str(it), width=88, subsequent_indent="      "))
    if not items:
        print("    (none)")

def respond(user_text):
    """One demo turn: feed a user message through the SAME pipeline + tracker, show the mechanism."""
    global turn_index
    rec = pipeline.step(history, tracker, Turn("user", user_text), "demo", turn_index)
    turn_index += 1
    print("=" * 72)
    print("You:  " + textwrap.fill(user_text, width=88, subsequent_indent="      "))
    print("Bot:  " + textwrap.fill(rec.bot_text, width=88, subsequent_indent="      "))
    print()
    _show("evidence retrieved for this reply", rec.passages_in_prompt)
    _show("claim log so far (persists across turns)", list(tracker.claims))
    if len(rec.candidates) > 1:
        print(f"  reranked {len(rec.candidates)} sampled candidates; kept the least self-contradictory")
    return rec

## Scripted mini-debate

Runs without typing \u2014 good for rehearsal. Watch the claim log accumulate across the three turns.

In [ ]:
# A scripted mini-debate -- runs top to bottom with no typing, so it is safe for rehearsal
# (and for a non-interactive execnotebook run). Watch the claim log grow across the three turns.
for msg in [
    "I think social media does more harm than good, especially for teenagers.",
    "But isn't it also how people find community and support?",
    "Still, the mental-health research looks pretty damning to me.",
]:
    respond(msg)

## Live Q&A (interactive)

Take questions from the audience. `quit` / `exit` / blank line to stop.

In [ ]:
# Live Q&A: type a message and press Enter; the bot answers using the same persistent tracker.
# Type 'quit', 'exit', or an empty line to stop. (Interactive input -- run in mynotebook, not execnotebook.)
print("Type a message and press Enter. 'quit' / 'exit' / blank line to stop.\n")
while True:
    try:
        msg = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if msg.lower() in {"quit", "exit", ""}:
        print("(demo ended)")
        break
    respond(msg)

## Reset

Run this to start a fresh conversation (clears the claim log and history).

In [ ]:
# Start a fresh conversation: new claim log and empty history (same loaded pipeline).
tracker = ClaimTracker(max_claims=cfg["claims"]["max_claims"], dedup_threshold=cfg["claims"]["dedup_threshold"])
history, turn_index = [], 0
print("Conversation reset: fresh claim log and history.")